# EH-DLF — Fusion Ablation: Average

**All three streams (Xception / FFT / SRM) are identical to EH-DLF.**
Only the fusion mechanism changes: instead of cross-attention, the three
256-d embeddings are element-wise averaged to a single 256-d vector.

```
x_rgb (256) ─┐
x_fft (256) ─┼─ Average (element-wise) → 256 → Dense(256,ReLU,Drop0.4)
x_srm (256) ─┘                                 → Dense(64,ReLU,Drop0.3)
                                                → Dense(1,sigmoid)
```

**EH-DLF reference (cross-attention):**
- FF++ AUC : 0.9638
- Celeb AUC: 0.7432

---
## Cell 1 — Imports & Constants

In [1]:
print("EH-DLF Fusion Ablation — AVERAGE")

import numpy as np
import os, cv2, json, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import Xception
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau,
    ModelCheckpoint, LearningRateScheduler
)
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Constants ──────────────────────────────────────────────────────────────
IMG_SIZE      = 299
BATCH_SIZE    = 32
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 20
AUTOTUNE      = tf.data.AUTOTUNE

# ── Paths — update to your Kaggle dataset slug ─────────────────────────────
DATASET_ROOT = "/kaggle/input/datasets/jfaisal/deepfake-dataset"
FF_TRAIN_DIR = os.path.join(DATASET_ROOT, "ff_train")
FF_VAL_DIR   = os.path.join(DATASET_ROOT, "ff_val")
FF_TEST_DIR  = os.path.join(DATASET_ROOT, "ff_test")
CELEB_DIR    = os.path.join(DATASET_ROOT, "celeb_test")

CKPT_DIR = "/kaggle/working/ckpts_avg"
FIG_DIR  = "/kaggle/working/figures_avg"
for d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# ── GPU ────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass

print(f"TF: {tf.__version__} | GPUs: {len(gpus)}")
print(f"IMG_SIZE={IMG_SIZE} | BATCH={BATCH_SIZE}"
      f" | P1={EPOCHS_PHASE1} | P2={EPOCHS_PHASE2}")
print("✓ Constants ready")

EH-DLF Fusion Ablation — AVERAGE


2026-04-21 18:34:50.406938: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776796490.797036      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776796490.911757      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776796491.878984      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776796491.879047      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776796491.879050      23 computation_placer.cc:177] computation placer alr

TF: 2.19.0 | GPUs: 2
IMG_SIZE=299 | BATCH=32 | P1=10 | P2=20
✓ Constants ready


---
## Cell 2 — Data Pipeline

In [2]:
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")


def make_dataset(directory, augment_flag=False):
    raw = tf.keras.utils.image_dataset_from_directory(
        directory, seed=SEED,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="binary")
    class_names = raw.class_names
    if augment_flag:
        raw = (raw
               .map(lambda x, y: (augment(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
               .shuffle(2000, reshuffle_each_iteration=True))
    return raw.prefetch(AUTOTUNE), class_names


train_ds,    train_cls = make_dataset(FF_TRAIN_DIR, augment_flag=True)
val_ds,      _         = make_dataset(FF_VAL_DIR,   augment_flag=False)
fftest_ds,   _         = make_dataset(FF_TEST_DIR,  augment_flag=False)
celebtest_ds,_         = make_dataset(CELEB_DIR,    augment_flag=False)

print("Class names:", train_cls, "→ fake=0, real=1")
assert train_cls == ['fake', 'real'], f"Unexpected: {train_cls}"

lbl    = np.concatenate([y.numpy() for _, y in train_ds])
n_fake = int((lbl == 0).sum())
n_real = int((lbl == 1).sum())
print(f"Train: {n_fake:,} fake  {n_real:,} real  (total {n_fake+n_real:,})")
print("✓ Data pipeline ready")

I0000 00:00:1776796532.565279      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1776796532.571788      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 14000 files belonging to 2 classes.
Found 3000 files belonging to 2 classes.
Found 3000 files belonging to 2 classes.
Found 6000 files belonging to 2 classes.
Class names: ['fake', 'real'] → fake=0, real=1
Train: 7,000 fake  7,000 real  (total 14,000)
✓ Data pipeline ready


---
## Cell 3 — Custom Layers & Loss (identical to EH-DLF)

In [3]:
class XceptionPreprocess(layers.Layer):
    def call(self, x):
        return tf.keras.applications.xception.preprocess_input(x)
    def get_config(self):
        return super().get_config()


class SRMFilter(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        k1 = np.array([[ 0, 0, 0, 0, 0],
                        [ 0,-1, 2,-1, 0],
                        [ 0, 2,-4, 2, 0],
                        [ 0,-1, 2,-1, 0],
                        [ 0, 0, 0, 0, 0]], dtype=np.float32) / 4.0
        k2 = np.array([[-1, 2,-2, 2,-1],
                        [ 2,-6, 8,-6, 2],
                        [-2, 8,-12,8,-2],
                        [ 2,-6, 8,-6, 2],
                        [-1, 2,-2, 2,-1]], dtype=np.float32) / 12.0
        k3 = np.array([[ 0, 0, 0, 0, 0],
                        [ 0,-1, 0, 1, 0],
                        [ 0, 0, 0, 0, 0],
                        [ 0, 1, 0,-1, 0],
                        [ 0, 0, 0, 0, 0]], dtype=np.float32) / 2.0
        self.k1 = tf.constant(k1[:,:,np.newaxis,np.newaxis], dtype=tf.float32)
        self.k2 = tf.constant(k2[:,:,np.newaxis,np.newaxis], dtype=tf.float32)
        self.k3 = tf.constant(k3[:,:,np.newaxis,np.newaxis], dtype=tf.float32)
    def call(self, x):
        gray = tf.image.rgb_to_grayscale(x / 255.0)
        r1   = tf.nn.conv2d(gray, self.k1, strides=1, padding='SAME')
        r2   = tf.nn.conv2d(gray, self.k2, strides=1, padding='SAME')
        r3   = tf.nn.conv2d(gray, self.k3, strides=1, padding='SAME')
        return tf.clip_by_value(tf.concat([r1, r2, r3], axis=-1), -1.0, 1.0)
    def get_config(self):
        return super().get_config()


class SRMStream(layers.Layer):
    def __init__(self, embed_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.srm   = SRMFilter()
        self.conv1 = layers.Conv2D(64,  3, activation='relu', padding='same')
        self.bn1   = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(128, 3, strides=2, activation='relu', padding='same')
        self.bn2   = layers.BatchNormalization()
        self.conv3 = layers.Conv2D(embed_dim, 3, strides=2, activation='relu', padding='same')
        self.bn3   = layers.BatchNormalization()
        self.gap   = layers.GlobalAveragePooling2D()
    def call(self, x, training=False):
        h = self.srm(x)
        h = self.bn1(self.conv1(h), training=training)
        h = self.bn2(self.conv2(h), training=training)
        h = self.bn3(self.conv3(h), training=training)
        return self.gap(h)
    def get_config(self):
        cfg = super().get_config()
        cfg["embed_dim"] = self.embed_dim
        return cfg


class FrequencyStream(layers.Layer):
    def __init__(self, embed_dim=256, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.conv1 = layers.Conv2D(64,  3, activation='relu', padding='same')
        self.bn1   = layers.BatchNormalization()
        self.conv2 = layers.Conv2D(128, 3, strides=2, activation='relu', padding='same')
        self.bn2   = layers.BatchNormalization()
        self.conv3 = layers.Conv2D(embed_dim, 3, strides=2, activation='relu', padding='same')
        self.bn3   = layers.BatchNormalization()
        self.gap   = layers.GlobalAveragePooling2D()
    def call(self, x, training=False):
        gray    = tf.image.rgb_to_grayscale(x / 255.0)
        gray    = tf.squeeze(gray, axis=-1)
        fft     = tf.signal.fft2d(tf.cast(gray, tf.complex64))
        fft     = tf.signal.fftshift(fft)
        mag     = tf.math.log1p(tf.abs(fft))
        mn      = tf.reduce_min(mag, axis=[1, 2], keepdims=True)
        mx      = tf.reduce_max(mag, axis=[1, 2], keepdims=True)
        mag     = (mag - mn) / (mx - mn + 1e-8)
        fft_img = tf.stack([mag, mag, mag], axis=-1)
        h = self.bn1(self.conv1(fft_img), training=training)
        h = self.bn2(self.conv2(h),       training=training)
        h = self.bn3(self.conv3(h),       training=training)
        return self.gap(h)
    def get_config(self):
        cfg = super().get_config()
        cfg["embed_dim"] = self.embed_dim
        return cfg


class LabelSmoothedBCE(tf.keras.losses.Loss):
    def __init__(self, smoothing=0.05, **kwargs):
        super().__init__(**kwargs)
        self.smoothing = smoothing
    def call(self, y_true, y_pred):
        y_true = y_true * (1 - self.smoothing) + 0.5 * self.smoothing
        return tf.keras.losses.binary_crossentropy(y_true, y_pred)
    def get_config(self):
        cfg = super().get_config()
        cfg["smoothing"] = self.smoothing
        return cfg


CUSTOM_OBJECTS = {
    "XceptionPreprocess": XceptionPreprocess,
    "SRMFilter":          SRMFilter,
    "SRMStream":          SRMStream,
    "FrequencyStream":    FrequencyStream,
    "LabelSmoothedBCE":   LabelSmoothedBCE,
}
print("✓ All custom layers defined")

✓ All custom layers defined


---
## Cell 4 — Build Average Fusion Model

Streams A / B / C are byte-for-byte identical to EH-DLF.
The only change: element-wise average of all three 256-d embeddings,
then standard classification head — no attention mechanism.

In [4]:
def build_model(img_size=299):
    inp = layers.Input(shape=(img_size, img_size, 3), name="input")

    # ── Stream A: Xception Spatial (identical to EH-DLF) ──────────────
    x_pre    = XceptionPreprocess(name="xception_preprocess")(inp)
    backbone = Xception(weights='imagenet', include_top=False,
                        input_shape=(img_size, img_size, 3))
    backbone.trainable = False
    x_rgb = backbone(x_pre, training=False)
    x_rgb = layers.GlobalAveragePooling2D(name="xception_gap")(x_rgb)
    x_rgb = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="xception_proj")(x_rgb)
    x_rgb = layers.Dropout(0.3, name="drop_rgb")(x_rgb)

    # ── Stream B: FFT Frequency (identical to EH-DLF) ─────────────────
    x_fft = FrequencyStream(embed_dim=256, name="fft_stream")(inp)
    x_fft = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="fft_proj")(x_fft)
    x_fft = layers.Dropout(0.3, name="drop_fft")(x_fft)

    # ── Stream C: SRM Steganalytic (identical to EH-DLF) ──────────────
    x_srm = SRMStream(embed_dim=256, name="srm_stream")(inp)
    x_srm = layers.Dense(256, activation='relu',
                          kernel_regularizer=l2(1e-4),
                          name="srm_proj")(x_srm)
    x_srm = layers.Dropout(0.3, name="drop_srm")(x_srm)

    # ── AVERAGE FUSION: element-wise mean of all three streams ─────────
    fused = layers.Average(name="avg_fusion")([x_rgb, x_fft, x_srm])
    # fused.shape = (batch, 256)  — equal weight to all three streams

    # ── Classification head (identical to EH-DLF head) ────────────────
    out = layers.Dense(256, activation='relu',
                        kernel_regularizer=l2(1e-4),
                        name="head_dense1")(fused)
    out = layers.Dropout(0.4, name="drop_head1")(out)
    out = layers.Dense(64, activation='relu',
                        name="head_dense2")(out)
    out = layers.Dropout(0.3, name="drop_head2")(out)
    output = layers.Dense(1, activation='sigmoid', name="output")(out)

    return Model(inputs=inp, outputs=output,
                 name="EH_DLF_AvgFusion")


model = build_model()
model.summary()

total     = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"\n✓ Total params     : {total:,}")
print(f"✓ Trainable params : {trainable:,}  (Phase 1 — backbone frozen)")
print()
print("NOTE: Average model has fewer params than EH-DLF (no cross-attn).")
print("      This is expected — ablating the entire attention mechanism.")

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "EH_DLF_AvgFusion"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 299, 299,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_preprocess │ (None, 299, 299,  │          0 │ input[0][0]       │
│ (XceptionPreproces… │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception            │ (None, 10, 10,    │ 20,861,480 │ xception_preproc… │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_gap        │ (None, 2048)      │          0 │ xception[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_stream          │ (None, 256)       │    372,608 │ input[0][0]       │
│ (FrequencyStream)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ srm_stream          │ (None, 256)       │    372,608 │ input[0][0]       │
│ (SRMStream)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ xception_proj       │ (None, 256)       │    524,544 │ xception_gap[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fft_proj (Dense)    │ (None, 256)       │     65,792 │ fft_stream[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ srm_proj (Dense)    │ (None, 256)       │     65,792 │ srm_stream[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_rgb (Dropout)  │ (None, 256)       │          0 │ xception_proj[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_fft (Dropout)  │ (None, 256)       │          0 │ fft_proj[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_srm (Dropout)  │ (None, 256)       │          0 │ srm_proj[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ avg_fusion          │ (None, 256)       │          0 │ drop_rgb[0][0],   │
│ (Average)           │                   │            │ drop_fft[0][0],   │
│                     │                   │            │ drop_srm[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense1 (Dense) │ (None, 256)       │     65,792 │ avg_fusion[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_head1          │ (None, 256)       │          0 │ head_dense1[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ head_dense2 (Dense) │ (None, 64)        │     16,448 │ drop_head1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_head2          │ (None, 64)        │          0 │ head_dense2[0][0] │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │         65 │ drop_head2[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 22,345,129 (85.24 MB)

 Trainable params: 1,481,857 (5.65 MB)

 Non-trainable params: 20,863,272 (79.59 MB)


✓ Total params     : 22,345,129
✓ Trainable params : 1,481,857  (Phase 1 — backbone frozen)

NOTE: Average model has fewer params than EH-DLF (no cross-attn).
      This is expected — ablating the entire attention mechanism.


---
## Cell 5 — Callbacks

In [5]:
ckpt_p1 = os.path.join(CKPT_DIR, "avg_p1.keras")
ckpt_p2 = os.path.join(CKPT_DIR, "avg_p2.keras")

cb_ckpt_p1   = ModelCheckpoint(ckpt_p1, monitor='val_accuracy',
                                save_best_only=True, verbose=1)
cb_ckpt_p2   = ModelCheckpoint(ckpt_p2, monitor='val_accuracy',
                                save_best_only=True, verbose=1)
cb_reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.4,
                                  patience=3, min_lr=1e-7, verbose=1)
cb_early_p1  = EarlyStopping(monitor='val_accuracy', patience=6,
                               restore_best_weights=True, verbose=1)
cb_early_p2  = EarlyStopping(monitor='val_accuracy', patience=10,
                               restore_best_weights=True, verbose=1)

def cosine_lr(epoch):
    start    = EPOCHS_PHASE1
    progress = np.clip((epoch - start) / max(EPOCHS_PHASE2, 1), 0, 1)
    return float(1e-7 + 0.5 * (1e-5 - 1e-7) * (1 + np.cos(np.pi * progress)))

cb_cosine = LearningRateScheduler(cosine_lr, verbose=0)
print("✓ Callbacks ready")

✓ Callbacks ready


---
## Cell 6 — Phase 1 Training (Frozen Backbone)

In [6]:
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=LabelSmoothedBCE(0.05),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

print("=" * 60)
print("PHASE 1 — Average Fusion | Backbone FROZEN")
print("=" * 60)
print(f"  Trainable params: "
      f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}")

history_p1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=[cb_ckpt_p1, cb_reduce_lr, cb_early_p1],
    verbose=1
)

best_p1 = max(history_p1.history['val_accuracy'])
print(f"\n✓ Phase 1 complete  |  Best val accuracy: {best_p1*100:.2f}%")

PHASE 1 — Average Fusion | Backbone FROZEN
  Trainable params: 1,481,857
Epoch 1/10


I0000 00:00:1776796917.646152      69 service.cc:152] XLA service 0x7893280bc050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776796917.646196      69 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776796917.646200      69 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776796919.818424      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-21 18:42:08.551994: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=2} for conv %cudnn-conv.93 = (f32[32,128,147,147]{3,2,1,0}, u8[0]{0}) custom-call(f32[32,128,147,147]{3,2,1,0} %bitcast.22346, f32[128,1,3,3]{3,2,1,0} %bitcast.22350), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=128, custom_call_target="__cudnn$convForward", metadata={op_type="DepthwiseConv2dNative" op_name="EH_DLF_AvgFusion_1/xception_1/block2_sepconv2_1/separable

298/438 ━━━━━━━━━━━━━━━━━━━━ 2:06 903ms/step - accuracy: 0.5245 - auc: 0.5312 - loss: 0.8013 - precision: 0.5161 - recall: 0.4980

2026-04-21 18:47:32.805823: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:47:33.028797: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:47:35.844705: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:47:36.037483: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:47:37.783796: E external/local_xla/xla/stream_

438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 996ms/step - accuracy: 0.5304 - auc: 0.5398 - loss: 0.7929 - precision: 0.5247 - recall: 0.5022

2026-04-21 18:51:19.714227: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:51:19.979783: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:51:23.585960: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:51:23.805263: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 18:51:26.670160: E external/local_xla/xla/stream_


Epoch 1: val_accuracy improved from -inf to 0.57333, saving model to /kaggle/working/ckpts_avg/avg_p1.keras
438/438 ━━━━━━━━━━━━━━━━━━━━ 721s 1s/step - accuracy: 0.5305 - auc: 0.5399 - loss: 0.7929 - precision: 0.5247 - recall: 0.5023 - val_accuracy: 0.5733 - val_auc: 0.6495 - val_loss: 0.7263 - val_precision: 0.5467 - val_recall: 0.8587 - learning_rate: 0.0010
Epoch 2/10
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 906ms/step - accuracy: 0.6089 - auc: 0.6443 - loss: 0.7127 - precision: 0.6164 - recall: 0.6024
Epoch 2: val_accuracy did not improve from 0.57333
438/438 ━━━━━━━━━━━━━━━━━━━━ 559s 999ms/step - accuracy: 0.6090 - auc: 0.6444 - loss: 0.7126 - precision: 0.6164 - recall: 0.6023 - val_accuracy: 0.5000 - val_auc: 0.6845 - val_loss: 1.1600 - val_precision: 0.5000 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 3/10
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 901ms/step - accuracy: 0.6382 - auc: 0.6913 - loss: 0.6733 - precision: 0.6361 - recall: 0.6538
Epoch 3: val_accuracy did not improve from 0.57

---
## Cell 7 — Phase 2 Training (Fine-Tune Top-40 Xception Layers)

In [7]:
model = tf.keras.models.load_model(ckpt_p1, custom_objects=CUSTOM_OBJECTS)

backbone = model.get_layer("xception")
backbone.trainable = True
UNFREEZE_FROM = len(backbone.layers) - 40
for layer in backbone.layers[:UNFREEZE_FROM]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

n_unfrozen = sum(1 for l in backbone.layers if l.trainable)
trainable  = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f"Xception layers unfrozen: {n_unfrozen}/{len(backbone.layers)}")
print(f"Trainable params        : {trainable:,}")

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=LabelSmoothedBCE(0.05),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name='accuracy'),
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ]
)

print("=" * 60)
print("PHASE 2 — Average Fusion | Fine-tuning top 40 layers")
print("=" * 60)

history_p2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1 + EPOCHS_PHASE2,
    initial_epoch=history_p1.epoch[-1] + 1,
    callbacks=[cb_ckpt_p2, cb_cosine, cb_reduce_lr, cb_early_p2],
    verbose=1
)

best_p2 = max(history_p2.history['val_accuracy'])
print(f"\n✓ Phase 2 complete  |  Best val accuracy: {best_p2*100:.2f}%")
model.save(os.path.join(CKPT_DIR, "avg_final.keras"))
print("✓ Final model saved")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'fft_stream', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'srm_stream', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Xception layers unfrozen: 28/132
Trainable params        : 12,013,273
PHASE 2 — Average Fusion | Fine-tuning top 40 layers
Epoch 8/30


2026-04-21 19:49:36.715535: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:49:36.870886: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:49:38.555272: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:49:38.695754: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:49:39.438221: E external/local_xla/xla/stream_

168/438 ━━━━━━━━━━━━━━━━━━━━ 4:44 1s/step - accuracy: 0.6284 - auc: 0.6720 - loss: 0.7059 - precision: 0.6480 - recall: 0.5832

2026-04-21 19:53:00.140417: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:53:00.284954: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:53:00.854058: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:53:00.993195: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-21 19:53:01.136588: E external/local_xla/xla/stream_

438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6588 - auc: 0.7139 - loss: 0.6794 - precision: 0.6771 - recall: 0.6172
Epoch 8: val_accuracy improved from -inf to 0.79733, saving model to /kaggle/working/ckpts_avg/avg_p2.keras
438/438 ━━━━━━━━━━━━━━━━━━━━ 714s 1s/step - accuracy: 0.6589 - auc: 0.7140 - loss: 0.6793 - precision: 0.6771 - recall: 0.6173 - val_accuracy: 0.7973 - val_auc: 0.8832 - val_loss: 0.5292 - val_precision: 0.7866 - val_recall: 0.8160 - learning_rate: 1.0000e-05
Epoch 9/30
438/438 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8080 - auc: 0.8855 - loss: 0.5137 - precision: 0.8143 - recall: 0.8021
Epoch 9: val_accuracy improved from 0.79733 to 0.85933, saving model to /kaggle/working/ckpts_avg/avg_p2.keras
438/438 ━━━━━━━━━━━━━━━━━━━━ 619s 1s/step - accuracy: 0.8081 - auc: 0.8856 - loss: 0.5136 - precision: 0.8144 - recall: 0.8021 - val_accuracy: 0.8593 - val_auc: 0.9456 - val_loss: 0.4051 - val_precision: 0.8722 - val_recall: 0.8420 - learning_rate: 1.0000e-05


---
## Cell 8 — Training Curves

In [8]:
ep1 = list(range(1, len(history_p1.history['accuracy']) + 1))
ep2 = list(range(len(ep1) + 1,
                 len(ep1) + len(history_p2.history['accuracy']) + 1))
pb  = ep1[-1] + 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ['accuracy', 'auc'], ['Accuracy', 'AUC']):
    ax.plot(ep1, history_p1.history[metric],
            '#2166ac', lw=1.8, ls='--', label='P1 Train')
    ax.plot(ep1, history_p1.history[f'val_{metric}'],
            '#d6604d', lw=1.8, ls='--', label='P1 Val')
    ax.plot(ep2, history_p2.history[metric],
            '#2166ac', lw=2.2,           label='P2 Train')
    ax.plot(ep2, history_p2.history[f'val_{metric}'],
            '#d6604d', lw=2.2,           label='P2 Val')
    ax.axvline(pb, color='grey', lw=1.2, ls=':')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.set_title(f'Average Fusion — {title}', fontweight='bold')
    ax.set_ylim(0.45, 1.01)
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.35)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'avg_curves.pdf'), dpi=300)
plt.savefig(os.path.join(FIG_DIR, 'avg_curves.png'), dpi=300)
plt.show()
print("✓ Saved: avg_curves")

✓ Saved: avg_curves


---
## Cell 9 — Load Final Model

In [9]:
model = tf.keras.models.load_model(
    os.path.join(CKPT_DIR, "avg_final.keras"),
    custom_objects=CUSTOM_OBJECTS
)
print("✓ Final model loaded")


def evaluate_and_plot(ds, dataset_name, tag, cmap='Blues'):
    print(f"\n{'─'*55}")
    print(f"  Evaluating: {dataset_name}")
    print(f"{'─'*55}")

    res = model.evaluate(ds, verbose=1)
    print(f"  Accuracy  : {res[1]*100:.2f}%")
    print(f"  AUC       : {res[2]:.4f}")
    print(f"  Precision : {res[3]:.4f}")
    print(f"  Recall    : {res[4]:.4f}")

    y_true, y_prob = [], []
    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        y_true.extend(labels.numpy().flatten())
        y_prob.extend(probs.flatten())
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    print("\n" + classification_report(
        y_true, y_pred, target_names=['Fake', 'Real']))
    auc_val = roc_auc_score(y_true, y_prob)
    print(f"  ROC-AUC: {auc_val:.4f}")

    # Confusion matrix
    cm     = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    acc    = (cm[0,0] + cm[1,1]) / cm.sum()

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cm, annot=False, cmap=cmap, ax=ax, cbar=True,
                square=True, linewidths=0.5,
                xticklabels=['Fake','Real'], yticklabels=['Fake','Real'])
    for i in range(2):
        for j in range(2):
            col = 'white' if cm_pct[i,j] > 50 else 'black'
            ax.text(j+0.5, i+0.38, f"{cm[i,j]:,}",
                    ha='center', va='center',
                    fontsize=15, fontweight='bold', color=col)
            ax.text(j+0.5, i+0.62, f"({cm_pct[i,j]:.1f}%)",
                    ha='center', va='center', fontsize=9, color=col)
    ax.set_title(f'{dataset_name}\nAcc={acc*100:.2f}%  AUC={auc_val:.4f}',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=10)
    ax.set_xlabel('Predicted Label', fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'{tag}_cm.pdf'), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f'{tag}_cm.png'), dpi=300)
    plt.show()
    print(f"  ✓ Saved: {tag}_cm")

    # ROC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, lw=2.5,
            label=f'{dataset_name} (AUC={auc_val:.4f})')
    ax.plot([0,1],[0,1],'k--', lw=1.2, alpha=0.6)
    ax.set_xlabel('FPR', fontsize=11); ax.set_ylabel('TPR', fontsize=11)
    ax.set_title(f'ROC — {dataset_name}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f'{tag}_roc.pdf'), dpi=300)
    plt.savefig(os.path.join(FIG_DIR, f'{tag}_roc.png'), dpi=300)
    plt.show()
    print(f"  ✓ Saved: {tag}_roc")

    report = classification_report(y_true, y_pred, output_dict=True)
    return {'name': dataset_name, 'tag': tag,
            'acc': acc, 'auc': auc_val,
            'f1': report['weighted avg']['f1-score'],
            'cm': cm, 'fpr': fpr, 'tpr': tpr}


print("✓ Evaluation engine ready")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'fft_stream', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:421: UserWarning: `build()` was called on layer 'srm_stream', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


✓ Final model loaded
✓ Evaluation engine ready


---
## Cell 10 — Evaluate on FF++ C23 Test Set

In [10]:
r_ff = evaluate_and_plot(
    fftest_ds,
    dataset_name="Average — FF++ C23",
    tag="avg_ffpp",
    cmap="Blues"
)


───────────────────────────────────────────────────────
  Evaluating: Average — FF++ C23
───────────────────────────────────────────────────────
94/94 ━━━━━━━━━━━━━━━━━━━━ 68s 577ms/step - accuracy: 0.9034 - auc: 0.9721 - loss: 0.3498 - precision: 0.8691 - recall: 0.9459
  Accuracy  : 91.10%
  AUC       : 0.9744
  Precision : 0.8817
  Recall    : 0.9493

              precision    recall  f1-score   support

        Fake       0.95      0.87      0.91      1500
        Real       0.88      0.95      0.92      1500

    accuracy                           0.91      3000
   macro avg       0.92      0.91      0.91      3000
weighted avg       0.92      0.91      0.91      3000

  ROC-AUC: 0.9747
  ✓ Saved: avg_ffpp_cm
  ✓ Saved: avg_ffpp_roc


---
## Cell 11 — Evaluate on Celeb-DF v2 (Zero-Shot)

In [11]:
r_celeb = evaluate_and_plot(
    celebtest_ds,
    dataset_name="Average — Celeb-DF v2 (zero-shot)",
    tag="avg_celeb",
    cmap="Oranges"
)


───────────────────────────────────────────────────────
  Evaluating: Average — Celeb-DF v2 (zero-shot)
───────────────────────────────────────────────────────
188/188 ━━━━━━━━━━━━━━━━━━━━ 98s 520ms/step - accuracy: 0.6389 - auc: 0.7437 - loss: 0.9984 - precision: 0.5856 - recall: 0.8937
  Accuracy  : 64.02%
  AUC       : 0.7463
  Precision : 0.5924
  Recall    : 0.8990

              precision    recall  f1-score   support

        Fake       0.79      0.38      0.51      3000
        Real       0.59      0.90      0.71      3000

    accuracy                           0.64      6000
   macro avg       0.69      0.64      0.61      6000
weighted avg       0.69      0.64      0.61      6000

  ROC-AUC: 0.7465
  ✓ Saved: avg_celeb_cm
  ✓ Saved: avg_celeb_roc


---
## Cell 12 — Final Summary

In [12]:
EHDLF_FF_AUC    = 0.9638
EHDLF_CELEB_AUC = 0.7432

print("\n" + "="*65)
print("  AVERAGE FUSION — FINAL RESULTS")
print("="*65)
print(f"  {'Model':<28} {'FF++ AUC':>10} {'Celeb AUC':>12} {'ΔGap':>8}")
print("─"*65)
print(f"  {'EH-DLF (Cross-attention)':<28} "
      f"{EHDLF_FF_AUC:>10.4f} "
      f"{EHDLF_CELEB_AUC:>12.4f} "
      f"{abs(EHDLF_FF_AUC-EHDLF_CELEB_AUC):>8.4f}")
print(f"  {'Average (ours)':<28} "
      f"{r_ff['auc']:>10.4f} "
      f"{r_celeb['auc']:>12.4f} "
      f"{abs(r_ff['auc']-r_celeb['auc']):>8.4f}")
print("="*65)

delta_ff    = r_ff['auc']    - EHDLF_FF_AUC
delta_celeb = r_celeb['auc'] - EHDLF_CELEB_AUC
print(f"\n  Delta vs EH-DLF:  FF++={delta_ff:+.4f}  Celeb={delta_celeb:+.4f}")

if EHDLF_CELEB_AUC > r_celeb['auc']:
    print("\n  ✓ Cross-attention outperforms average fusion on Celeb-DF.")
    print("    Result confirms dynamic fusion > static equal weighting.")
else:
    print("\n  Average fusion matches/exceeds cross-attention on Celeb-DF.")
    print("    Report honestly. Both results are valid for the paper.")

print(f"\nFigures saved to: {FIG_DIR}")
for f in sorted(os.listdir(FIG_DIR)):
    sz = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
    print(f"  {f:<45} {sz:>6.1f} KB")
print("\n✓ Done. Download /kaggle/working/figures_avg/")


  AVERAGE FUSION — FINAL RESULTS
  Model                          FF++ AUC    Celeb AUC     ΔGap
─────────────────────────────────────────────────────────────────
  EH-DLF (Cross-attention)         0.9638       0.7432   0.2206
  Average (ours)                   0.9747       0.7465   0.2282

  Delta vs EH-DLF:  FF++=+0.0109  Celeb=+0.0033

  Average fusion matches/exceeds cross-attention on Celeb-DF.
    Report honestly. Both results are valid for the paper.

Figures saved to: /kaggle/working/figures_avg
  avg_celeb_cm.pdf                                26.9 KB
  avg_celeb_cm.png                               113.8 KB
  avg_celeb_roc.pdf                               33.1 KB
  avg_celeb_roc.png                              127.4 KB
  avg_curves.pdf                                  20.7 KB
  avg_curves.png                                 192.9 KB
  avg_ffpp_cm.pdf                                 23.2 KB
  avg_ffpp_cm.png                                104.1 KB
  avg_ffpp_roc.pdf        